In [8]:
# Load dan Integrasi Dataset untuk Fashion & Lifestyle dan F&B di Surabaya
import pandas as pd
import numpy as np

# Memuat Dataset
df_fashion = pd.read_csv('../dataset/data_produk.csv')
display(df_fashion.head())
print(f"Jumlah data fashion: {len(df_fashion)}")
df_gofood = pd.read_csv('../dataset/gofood_dataset.csv')
print(f"Jumlah data gofood: {len(df_gofood)}")
display(df_gofood.head())

# Standardisasi Data Fashion
df_fashion_clean = df_fashion[['shop_name', 'category', 'item_name', 'description', 'price']].copy()
df_fashion_clean.columns = ['nama_tenant', 'kategori', 'produk', 'deskripsi', 'harga']
df_fashion_clean['sektor'] = 'Fashion & Lifestyle'

# Standardisasi Data GoFood (Filter khusus area lokal)
df_gofood_sub = df_gofood[df_gofood['merchant_area'] == 'surabaya'].copy()
df_gofood_clean = df_gofood_sub[['merchant_name', 'category', 'product', 'description', 'price']].copy()
df_gofood_clean.columns = ['nama_tenant', 'kategori', 'produk', 'deskripsi', 'harga']
df_gofood_clean['sektor'] = 'F&B'

# Integrasi
df_all = pd.concat([df_fashion_clean, df_gofood_clean], ignore_index=True)
display(df_all.head())
df_all.shape

,item_name,description,category,price,colour,shop_name
0,Sepatu Sneakers,Sepatu Sneakers adalah pilihan yang sempurna u...,Sepatu,100000,Putih,Home Essentials
1,Tas Ransel Stylish,Tas Ransel Stylish memberikan gaya dan fungsi ...,Tas,200000,Merah,Sports World
2,Kemeja Flanel Modern,Kemeja Flanel Modern memberikan tampilan yang ...,Kemeja,150000,Hijau,Gourmet Delights
3,Celana Jeans Slim Fit,Celana Jeans Slim Fit adalah pilihan yang tepa...,Celana,250000,Biru,Gadget Hub
4,Topi Fedora Elegan,Topi Fedora Elegan memberikan sentuhan klasik ...,Topi,50000,Abu-abu,Gourmet Delights


Jumlah data fashion: 100
Jumlah data gofood: 45195


,merchant_name,merchant_area,category,display,product,price,discount_price,isDiscount,description
0,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Hot Almara Kopi (kopi Susu Gula Aren),20000.0,NaN,0,Sajian Kopi Susu Gula Aren Yang Berbeda Dari K...
1,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Ice Almara Kopi (kopi Susu Gula Aren),22000.0,NaN,0,Sajian Kopi Susu Gula Aren Yang Berbeda Dari K...
2,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Hot Millsis,20000.0,NaN,0,Sajian Susu Coklat Milo Dengan Racikan Khas 3 ...
3,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Ice Millsis,20000.0,NaN,0,Sajian Susu Coklat Milo Dengan Racikan Khas 3 ...
4,"330 Kopi, Ciledug",jakarta,Kopi/Minuman/Roti,Signature,Hot Millbro,22000.0,NaN,0,Sajian Susu Coklat Milo Plus Espresso Dengan R...


,nama_tenant,kategori,produk,deskripsi,harga,sektor
0,Home Essentials,Sepatu,Sepatu Sneakers,Sepatu Sneakers adalah pilihan yang sempurna u...,100000.0,Fashion & Lifestyle
1,Sports World,Tas,Tas Ransel Stylish,Tas Ransel Stylish memberikan gaya dan fungsi ...,200000.0,Fashion & Lifestyle
2,Gourmet Delights,Kemeja,Kemeja Flanel Modern,Kemeja Flanel Modern memberikan tampilan yang ...,150000.0,Fashion & Lifestyle
3,Gadget Hub,Celana,Celana Jeans Slim Fit,Celana Jeans Slim Fit adalah pilihan yang tepa...,250000.0,Fashion & Lifestyle
4,Gourmet Delights,Topi,Topi Fedora Elegan,Topi Fedora Elegan memberikan sentuhan klasik ...,50000.0,Fashion & Lifestyle


(15322, 6)

In [9]:
# Preprocessing dan Rekayasa Fitur
# Penanganan Missing Values
df_all['kategori'] = df_all['kategori'].fillna('')
df_all['produk'] = df_all['produk'].fillna('')
df_all['deskripsi'] = df_all['deskripsi'].fillna('')
df_all['harga'] = pd.to_numeric(df_all['harga'], errors='coerce').fillna(0)

# Agregasi Level Tenant
df_tenant = df_all.groupby('nama_tenant').agg({
    'kategori': lambda x: ' '.join(set(x.astype(str))),
    'produk': lambda x: ' '.join(x.astype(str)),
    'deskripsi': lambda x: ' '.join(x.astype(str)),
    'harga': 'mean',
    'sektor': 'first'
}).reset_index()
display(df_tenant.head())

# Pembuatan Variabel Sintetik
df_tenant['metadata'] = df_tenant['kategori'] + " " + df_tenant['produk'] + " " + df_tenant['deskripsi']
df_tenant['metadata'] = df_tenant['metadata'].str.lower()
display(df_tenant[['nama_tenant', 'metadata']].head())


,nama_tenant,kategori,produk,deskripsi,harga,sektor
0,Mie Jeder,Bakmie,Mie Pedas Jeder Level 0 Mie Pedas Jeder Level ...,"Mie Original Tidak Pedas + Toping Ayam, Beef, ...",13000.000000,F&B
1,"369 Shanghai, Jemursari",Chinese,Bestdeal Noodle Bestdeal Sup Bestdeal rice Sio...,Song mie/ Mie Zha jiang +Xiaolongbao/Guo tie s...,66905.381818,F&B
2,AHA by Kudos,Minuman/Kopi/Pizza & pasta,Beef And Mushroom Shirataki Spicy Abon Cakalan...,"Rice, Beef Patty, Mushrooms, Chilli, Coconut M...",40652.857143,F&B
3,"AMPM SDA, Buduran",Minuman/Cepat saji/Kopi,Iced Freshpresso Iced Coffee Latte Iced Americ...,"Espresso, Susu, Dengan Sirup mint Nikmatnya ...",21454.545455,F&B
4,"Aiciro, BG Junction",Jajanan/Ayam & bebek/Minuman,Chicken Fillet Kentang Siomay Nugget Chicken S...,"Pilihan Rasa: Original, Chili, Barbeque // Lev...",15619.047619,F&B


,nama_tenant,metadata
0,Mie Jeder,bakmie mie pedas jeder level 0 mie pedas jeder...
1,"369 Shanghai, Jemursari",chinese bestdeal noodle bestdeal sup bestdeal ...
2,AHA by Kudos,minuman/kopi/pizza & pasta beef and mushroom s...
3,"AMPM SDA, Buduran",minuman/cepat saji/kopi iced freshpresso iced ...
4,"Aiciro, BG Junction",jajanan/ayam & bebek/minuman chicken fillet ke...


In [10]:
# Ekstraksi Fitur 
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi TF-IDF dengan penghapusan kata hubung bahasa Inggris
tfidf = TfidfVectorizer(stop_words='english')
print("Fitting TF-IDF Vectorizer...")
tfidf.fit(df_tenant['metadata'])

# Transformasi teks menjadi matriks vektor
tfidf_matrix = tfidf.fit_transform(df_tenant['metadata'])
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

Fitting TF-IDF Vectorizer...
TF-IDF Matrix Shape: (288, 5816)


In [11]:
# Kompputasi dan Implementasi Sistem Rekomendasi
from sklearn.metrics.pairwise import cosine_similarity
import pickle

def get_recommendations(user_query, price_limit=None, sektor_filter=None):
    # 1. Mengubah kueri pencarian menjadi vektor
    query_vec = tfidf.transform([user_query.lower()])
    
    # 2. Menghitung jarak kemiripan
    cosine_sim = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # 3. Menyimpan skor ke dataframe sementara
    results = df_tenant.copy()
    results['score'] = cosine_sim
    print("Results with Scores:\n", results[['nama_tenant', 'score']].head())
    
    # 4. Filter Heuristik: Batasan Harga
    if price_limit:
        results = results[results['harga'] <= price_limit]
        print(f"Results after Price Filter (<= {price_limit}):\n", results[['nama_tenant', 'harga']].head())
        
    # 5. Filter Heuristik: Kategori Sektor
    if sektor_filter and sektor_filter != 'Semua':
        results = results[results['sektor'] == sektor_filter]
        print(f"Results after Sector Filter ({sektor_filter}):\n", results[['nama_tenant', 'sektor']].head())
        
    # 6. Mengambil Top 10 rekomendasi
    final_results = results.sort_values(by='score', ascending=False).head(10)
    
    # Mengembalikan hasil
    return final_results[['nama_tenant', 'sektor', 'harga', 'score']]

# --- UJI COBA FUNGSI ---
print("Uji Coba: Mencari 'Makanan' di sektor F&B dengan budget 50rb")
# Gunakan print() jika dijalankan di terminal biasa, atau display() jika di Jupyter/Colab
print(get_recommendations("Makanan", price_limit=50000, sektor_filter="F&B"))


# --- TAHAP 6: EXPORT MODEL (PICKLE) ---
data_to_save = {
    'df_tenant': df_tenant,
    'tfidf_matrix': tfidf_matrix,
    'tfidf_model': tfidf
}

with open('model_rekomendasi.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)
    
print("\n[BERHASIL] File 'model_rekomendasi.pkl' berhasil diekspor!")

Uji Coba: Mencari 'Makanan' di sektor F&B dengan budget 50rb
Results with Scores:
                nama_tenant  score
0                Mie Jeder    0.0
1  369 Shanghai, Jemursari    0.0
2             AHA by Kudos    0.0
3        AMPM SDA, Buduran    0.0
4      Aiciro, BG Junction    0.0
Results after Price Filter (<= 50000):
                 nama_tenant         harga
0                 Mie Jeder  13000.000000
2              AHA by Kudos  40652.857143
3         AMPM SDA, Buduran  21454.545455
4       Aiciro, BG Junction  15619.047619
5  Amalia Cake, Wringinanom  21430.555556
Results after Sector Filter (F&B):
                 nama_tenant sektor
0                 Mie Jeder    F&B
2              AHA by Kudos    F&B
3         AMPM SDA, Buduran    F&B
4       Aiciro, BG Junction    F&B
5  Amalia Cake, Wringinanom    F&B
                                           nama_tenant sektor         harga  \
227  Seblak Dan Mie Ayam Kopi Santai Gkb, Jl Kalima...    F&B  18087.500000   
154              